Section 1: My Lane

Lane: Ranking Signal Analysis

I'm choosing this lane because this lane let me explore about observable signals including impressions, clicks, position, word count, freshness, engagement are actually associated with content performance before committing to a specific prediction target. The starter dataset and warehouse release both have strong depth for search-performance signals, GSC impression data covers the large majority of the daily fact rows  so there's enough evidence here to find real patterns rather than noise.

This is also a good starting lane because it's low-risk for Week 1: it doesn't require me to define a future-window label so I can build intuition about the data before locking in a harder-to-validate target. I may change toward Lane 2 (Refresh Scoring) or Lane 4 (CTR Scoring) later if a specific signal turns out to be strong enough to justify a ranking/scoring model.

Section 2: decision, action, cost of a wrong call

Question: Which observable content and search signals (e.g., average position, word count, freshness, impressions) are most associated with pages trending down in visibility, and can those signals give a content review team an early, lightweight way to flag pages worth a closer look?

Unit of analysis: Using one content page with content_id and using its most recent 90-day snapshot of signals.

Decision this improves: Which signals a content reviewer should weight when triaging their backlog by giving the team a short list of "signals worth watching" instead of reviewing every page with equal attention.

Action someone takes: A content strategist uses the identified signals as an informal secondary filter when deciding which pages to open first in their weekly review queue. For example, we can prioritizing pages with declining position trend and above-median impressions.

Cost of a wrong call: There are 2 types matter. 
    - A false signal wastes reviewer time on pages that didn't need attention.
    - A missed signal lets real decline go unnoticed longer, costing more traffic the longer it's missed.

Why data/ML support us in this case: With 500K+ content items and I belived that no one can manually scan for signal patterns at that scale. EDA can surface the observable signals actually correlate with movement by replacing guesswork with evidence, without claiming causation.





Section 3:

In [12]:
import pandas as pd
df = pd.read_csv("/Users/anhvuong/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")
df.shape

(30000, 44)

In [13]:
filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
filtered.shape

(30000, 44)

In [14]:
filtered["trend_direction"].value_counts(normalize=True)

trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64

In [15]:
filtered.groupby("trend_direction")["avg_position"].mean()

trend_direction
down      15.936305
flat      11.102431
new        9.912433
stable    16.332623
up        22.512739
Name: avg_position, dtype: float64

Section3: Summary for quick look at the data

The starter dataset, after applying the standard filters 
(`impressions_90d > 0` and `content_age_days >= 90`), contains 30,000 
content items across 44 columns and in my opinion I think it would be enough volume to look for patterns 
without worrying about small-sample noise at this stage.

The `trend_direction` label is heavily skewed toward decline: 54.2% of 
pages are labeled "down," compared to 19.9% stable, 14.6% up, 7.5% new, 
and 3.8% flat. Decline is the important outcome in this sample, not a 
rare event, which matters for how I frame the problem going forward.

Average position by trend direction is counter-intuitive: pages trending 
down sit at a better average position (~15.9) than pages trending up 
(~22.5). Two plausible explanations: 
    - Pages already ranking well have less room to climb and more room to fall (a ceiling effect)
    - Top-ranking pages may face more competitive pressure since they're worth 
contesting. 

I'm not sure both  explanation is clarify but the direction of this pattern isn't what I expected so I might not be useful signal for how I frame the question next.